# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The key fields show strong right-skew and heavy tails. For example, impressions have a median of 731 but a maximum of 517,715, while search volume has a median of 10 and a maximum of 74,000. Trend percentage is also highly variable, ranging from -100% to 44,900%, so extreme values should be treated carefully rather than assuming the mean represents a typical page.

In [12]:
!git clone https://github.com/FatimaNdeem/Flyrank-ml-internship..git

Cloning into 'Flyrank-ml-internship.'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 193 (delta 93), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 1.90 MiB | 17.34 MiB/s, done.
Resolving deltas: 100% (93/93), done.


In [13]:
import os
os.chdir("/content/Flyrank-ml-internship.")
print(os.getcwd())

/content/Flyrank-ml-internship.


In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


In [15]:
key_fields = [
    "impressions_90d",
    "clicks_90d",
    "search_volume",
    "avg_position",
    "trend_pct",
    "content_age_days"
]

df[key_fields].describe().T

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.0,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.0,7.00,4178.0
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.0,20.00,74000.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.8,22.30,245.0
trend_pct,26612.0,-4.785969,473.861780,-100.0,-62.6,-33.5,0.00,44900.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.0,333.00,564.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal #1** Impressions and clicks: The correlation between impressions and clicks is 0.696, showing a strong positive relationship in this dataset. This supports the expectation that pages receiving more impressions generally receive more clicks. Verdict: CONFIRMED.

In [16]:
signal_1 = df[["impressions_90d", "clicks_90d"]].corr().iloc[0, 1]

print("Correlation between impressions and clicks:", round(signal_1, 3))

Correlation between impressions and clicks: 0.696


**Signal #2** — Content age and impressions: The correlation between content age and impressions is -0.001, indicating almost no linear relationship in this dataset. This does not support the assumption that older content consistently has lower impressions. Verdict: MIXED.

In [17]:
print("Correlation between content age and impressions:",
      round(df["content_age_days"].corr(df["impressions_90d"]), 3))

Correlation between content age and impressions: -0.001


**Signal# 3** : Higher search volume should generally be associated with higher impressions.
Test: I measured the correlation between search volume and impressions.
Result: The correlation is 0.001, which is essentially zero.
Verdict: FALSE. In this dataset, search volume alone does not show a meaningful linear relationship with impressions. This suggests that other factors should be considered when prioritizing pages for refresh.

In [18]:
print(
    "Correlation between search volume and impressions:",
    round(df["search_volume"].corr(df["impressions_90d"]), 3)
)

Correlation between search volume and impressions: 0.001


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test — declining impression flag**

The declining flag marks a page when its last-30-day impressions are below 80% of its previous-30-day impressions. The flagged pages have average impressions of 941.6 in the last 30 days compared with 1,808.4 in the previous 30 days, and their average trend percentage is -58.1%. In contrast, unflagged pages have average last-30-day impressions of 2,006.1 and average trend percentage of +79.0%.

Verdict: CONFIRMED. The observed data supports the assumption behind the flag: flagged pages show substantially weaker recent impression performance and a negative average trend.

In [19]:
# Flag-linked test: declining impression flag
# Test whether pages flagged as declining actually have lower recent impressions.

df["is_declining_flag"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
)

flag_summary = df.groupby("is_declining_flag")[
    ["impressions_last_30d", "impressions_prev_30d", "trend_pct"]
].mean()

print(flag_summary)

print("\nNumber of flagged pages:",
      df["is_declining_flag"].sum())

print("Total pages:",
      len(df))

                   impressions_last_30d  impressions_prev_30d  trend_pct
is_declining_flag                                                       
False                       2006.126656           1753.083928  79.003179
True                         941.556635           1808.417661 -58.113830

Number of flagged pages: 16262
Total pages: 30000


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit shows that not all individual signals are equally useful for identifying refresh opportunities. The declining-impression flag is supported by the observed data, while content age and search volume alone show little linear relationship with impressions. In practice, a content team should use multiple signals together rather than relying on a single feature when prioritizing pages for refresh.

In [20]:
print("Signal audit completed.")
print("The declining-impression flag is supported by the observed data.")
print("Search volume and content age alone are weak signals.")

Signal audit completed.
The declining-impression flag is supported by the observed data.
Search volume and content age alone are weak signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.